# Estudo de Validação do Ataque ALGEN (SST-2)

Este notebook isola e valida o ataque ALGEN **antes** de qualquer grade de privacidade DPED (ε), respondendo:
1. **Oráculo (`oracle_target`)**: qual é o teto do decodificador congelado do ALGEN recebendo `Y_test` (`flan-t5-small`) diretamente no SST-2?
2. **ALGEN original (`bert_contextual`)**: o ataque reproduz o comportamento esperado quando usa embeddings contextuais (`last_hidden_state`), como no paper?
3. **BERT estático (`bert_pretrained`)**: quanto se perde ao usar a média de `word_embeddings`, e o ataque melhora com $n \gg d=768$, centralização (`--center`), normalização pós-alinhamento (`--normalize_aligned`) e ajuste de `ridge`?
4. **Controle positivo sem ALGEN (`direct_static_decode.py`)**: a recuperação direta por OMP sobre a tabela estática funciona melhor que o ALGEN?

Documentação: [validacao-algen-diagnostico.md](../docs/implementação/validacao-algen-diagnostico.md) e [plano_melhoria_metodologia.md](../docs/implementação/plano_melhoria_metodologia.md).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import subprocess

REPO_URL = 'https://github.com/otavio-hbf/DPED-Sistemas-Hibridos.git'
REPO = '/content/DPED-Sistemas-Hibridos'
BRANCH = 'robust-metodol'
if os.path.isdir(REPO):
    subprocess.run(['git', '-C', REPO, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, REPO], check=True)
subprocess.run(['git', '-C', REPO, 'status', '--short', '--branch'], check=True)

In [ ]:
%cd /content/DPED-Sistemas-Hibridos
!pip install -q -r requirements.txt evaluate rouge-score sacrebleu pot wandb

In [ ]:
import json
import os
import shutil
import subprocess
from datetime import datetime

os.chdir(REPO)

RUN_NAME = 'yiyic_mmarco_english_maxlength32_train150000_batch_size128_lr0.0001_wd0.0001_epochs100'
REPO_CKPT = os.path.join(REPO, 'algen/SIH-ALGEN-main/outputs/google_flan-t5-small', RUN_NAME)
DRIVE_CKPT = '/content/drive/MyDrive/SIH/algen/checkpoint'

with open(os.path.join(REPO_CKPT, 'training_args_and_best_models.json')) as f:
    _data = json.load(f)
_data['best_models'] = [
    [*entry[:-1], os.path.join(DRIVE_CKPT, os.path.basename(entry[-1]))]
    for entry in _data['best_models']
]
with open(os.path.join(DRIVE_CKPT, 'training_args_and_best_models.json'), 'w') as f:
    json.dump(_data, f, indent=4)

ALGEN_CKPT = DRIVE_CKPT
RUN_ID = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
DRIVE_RUN_DIR = os.path.join('/content/drive/MyDrive/SIH/algen_validation', RUN_ID)
LOCAL_OUT = 'results/algen_validation'
os.makedirs(DRIVE_RUN_DIR, exist_ok=True)
os.makedirs(LOCAL_OUT, exist_ok=True)

ATTACKER = 'algen/SIH-ALGEN-main/src/attacker_dped_sentence.py'
DIRECT_DECODER = 'algen/SIH-ALGEN-main/src/direct_static_decode.py'
TEST_SAMPLES = 200
SEED = 0

def persist_path(source, destination):
    if not os.path.exists(source):
        raise FileNotFoundError(f'Artefato esperado não encontrado: {source}')
    target = os.path.join(DRIVE_RUN_DIR, destination)
    os.makedirs(os.path.dirname(target), exist_ok=True)
    if os.path.isdir(source):
        shutil.copytree(source, target, dirs_exist_ok=True)
    else:
        shutil.copy2(source, target)
    return target

def run_attack_step(tag, source, train_samples, ridge=1.0, center=False, normalize_aligned=False):
    out_dir = os.path.join(LOCAL_OUT, tag)
    cmd = [
        'python', ATTACKER,
        '--checkpoint_path', ALGEN_CKPT,
        '--source', source,
        '--train_samples', str(train_samples),
        '--test_samples', str(TEST_SAMPLES),
        '--ridge', str(ridge),
        '--seed', str(SEED),
        '--run_kind', 'research',
        '--output_dir', out_dir,
    ]
    if center:
        cmd.append('--center')
    if normalize_aligned:
        cmd.append('--normalize_aligned')
    print(f'>>> [{tag}]', ' '.join(cmd))
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        print(res.stdout)
        print(res.stderr)
        raise RuntimeError(f'Falha em {tag}')
    persist_path(out_dir, tag)
    with open(os.path.join(out_dir, 'summary.json')) as f:
        return json.load(f)

print('ALGEN_CKPT exists:', os.path.isdir(ALGEN_CKPT), '| DRIVE_RUN_DIR:', DRIVE_RUN_DIR)

## Etapa 1: Oráculo do decodificador congelado (`oracle_target`)

Passa `Y_test` (`flan-t5-small` encoder mean-pool) diretamente para o decodificador Zenodo, sem alinhamento `X→Y`. Mede o teto absoluto do decodificador no SST-2 e o piso do preditor constante.

In [ ]:
oracle_summary = run_attack_step('01_oracle_target', source='oracle_target', train_samples=1000, ridge=1.0)
print('Oracle test_results:', oracle_summary['test_results'])
print('Constant baseline:', oracle_summary['constant_baseline'])
print('Anisotropy Y_test mean cos:', oracle_summary['align_metrics']['anisotropy_y_test_mean_cosine'])

## Etapa 2: Controle positivo ALGEN original (`bert_contextual`)

Usa `last_hidden_state` mean-pool do `bert-base-uncased` (ataque original do paper ALGEN) para `n ∈ {10, 100, 1000}`, além de `n=1000` com `--center --normalize_aligned`.

In [ ]:
contextual_summaries = []
for n in [10, 100, 1000]:
    s = run_attack_step(f'02_bert_contextual_n{n}', source='bert_contextual', train_samples=n, ridge=1.0)
    contextual_summaries.append((f'contextual_n{n}', s))
s_cn = run_attack_step('02_bert_contextual_n1000_center_norm', source='bert_contextual', train_samples=1000, ridge=1.0, center=True, normalize_aligned=True)
contextual_summaries.append(('contextual_n1000_center_norm', s_cn))

## Etapa 3: Diagnóstico do BERT estático (`bert_pretrained`)

1. Escala de `n` incluindo regime $n \gg d=768$: `{1, 10, 100, 1000, 5000, 10000}`.
2. Ablação de `--center` e `--normalize_aligned` em `n=1000` e `n=5000`.
3. Varredura de `ridge` $\lambda \in \{0.0, 10^{-4}, 10^{-2}, 10^{-1}, 1.0, 10.0\}$ em `n=5000` com `--center --normalize_aligned`.

In [ ]:
static_summaries = []
for n in [1, 10, 100, 1000, 5000, 10000]:
    s = run_attack_step(f'03_static_n{n}_r1.0', source='bert_pretrained', train_samples=n, ridge=1.0)
    static_summaries.append((f'static_n{n}_r1.0', s))

for n in [1000, 5000]:
    for center, norm, suffix in [(True, False, 'center'), (False, True, 'norm'), (True, True, 'center_norm')]:
        tag = f'03_static_n{n}_{suffix}'
        s = run_attack_step(tag, source='bert_pretrained', train_samples=n, ridge=1.0, center=center, normalize_aligned=norm)
        static_summaries.append((tag, s))

for lam in [0.0, 1e-4, 1e-2, 1e-1, 1.0, 10.0]:
    tag = f'03_static_n5000_center_norm_r{lam}'
    s = run_attack_step(tag, source='bert_pretrained', train_samples=5000, ridge=lam, center=True, normalize_aligned=True)
    static_summaries.append((tag, s))

## Etapa 4: Controle positivo sem ALGEN (`direct_static_decode.py`)

Decodificação direta por Orthogonal Matching Pursuit (OMP) sobre a tabela estática `bert-base-uncased`, para separar o limite do objeto estático do limite do alinhamento/decodificador ALGEN.

In [ ]:
omp_out = os.path.join(LOCAL_OUT, '04_direct_omp')
subprocess.run([
    'python', DIRECT_DECODER,
    '--test_samples', str(TEST_SAMPLES),
    '--max_steps', '20',
    '--min_gain', '0.001',
    '--output_dir', omp_out,
], check=True)
persist_path(omp_out, '04_direct_omp')
with open(os.path.join(omp_out, 'summary.json')) as f:
    omp_summary = json.load(f)
print('Direct OMP summary:', json.dumps(omp_summary, indent=2))

## Etapa 5: Tabela consolidada de diagnóstico

In [ ]:
import glob
import pandas as pd

rows = []
for path in sorted(glob.glob(os.path.join(LOCAL_OUT, '*', 'summary.json'))):
    tag = os.path.basename(os.path.dirname(path))
    if tag == '04_direct_omp':
        continue
    with open(path) as f:
        s = json.load(f)
    tr = s['test_results']
    cb = s['constant_baseline']['test_results']
    am = s['align_metrics']
    rows.append({
        'run': tag,
        'source': s['embedding_provenance'].get('source', 'bert_pretrained'),
        'n': s['train_samples'],
        'ridge': s['ridge'],
        'center': am.get('center', False),
        'norm': am.get('normalize_aligned', False),
        'rouge1': tr.get('rouge1'),
        'rougeL': tr.get('rougeL'),
        'bleu1': tr.get('bleu1'),
        'cos': am.get('X_Y_test_COS'),
        'distinct_preds': s.get('distinct_predictions'),
        'const_rouge1': cb.get('rouge1'),
        'delta_rouge1_vs_const': (tr.get('rouge1', 0) - cb.get('rouge1', 0)),
    })

df = pd.DataFrame(rows)
csv_path = os.path.join(LOCAL_OUT, 'validation_summary.csv')
df.to_csv(csv_path, index=False)
persist_path(csv_path, 'validation_summary.csv')
display(df)